# 04 数据增强消融（E3）与可选参数实验（E4）

**目标：** 固定骨干与微调策略，比较无随机增强、几何增强、几何与颜色组合增强，检验增强带来的变化。

**前置条件：** `selected_backbone` 和 `selected_finetuning` 均已依据验证结果填写；固定清单保持一致。

**输入：** 基础配置、增强候选、相同随机种子。

**输出：** `outputs/runs/e3_*/` 的真实实验记录；可选 E4 实验单独保存。本阶段只使用训练与验证集。

默认关闭训练。`none` 仍包含确定性缩放、张量转换与归一化。

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

def summarize_validation(plan_list):
    records = []
    for plan in plan_list:
        run_dir = PROJECT_ROOT / "outputs" / "runs" / plan["run_id"]
        history = read_csv(run_dir / "history.csv")
        if not history:
            continue
        config_path = run_dir / "config.yaml"
        actual_config = load_config(config_path) if config_path.exists() else plan["config"]
        monitor = actual_config["training"].get("monitor", "macro_f1")
        key = "val_" + monitor
        best = min(history, key=lambda row: float(row[key])) if monitor == "loss" else max(
            history, key=lambda row: float(row[key]))
        records.append({"run_id": plan["run_id"], "recorded_epochs": len(history),
                        "monitor": monitor, "best_epoch": int(best["epoch"]),
                        "val_loss": float(best["val_loss"]),
                        "val_accuracy": float(best["val_accuracy"]),
                        "val_macro_f1": float(best["val_macro_f1"]),
                        "total_training_seconds": float(history[-1]["elapsed_seconds"])})
    return records


## 1. 生成消融计划

增强的具体变换定义在 `src/data.py`。实验前阅读并记录参数范围；同一轮消融只改变增强策略，保持其他训练条件相同。

In [ ]:
RUN_TRAINING = False
selected_backbone = EXPERIMENTS.get("selected_backbone")
selected_finetuning = EXPERIMENTS.get("selected_finetuning")
plans = []
if not selected_backbone or not selected_finetuning:
    print("请先在 experiments.yaml 中填写 selected_backbone 和 selected_finetuning。")
else:
    for augmentation in EXPERIMENTS.get("augmentations", ["none", "basic", "combined"]):
        for seed in EXPERIMENTS.get("seeds", [BASE.get("seed", 42)]):
            config = deepcopy(BASE)
            config["model"]["name"] = selected_backbone
            config["model"]["finetuning"] = selected_finetuning
            config["augmentation"] = augmentation
            config["seed"] = int(seed)
            run_id = f"e3_{selected_backbone}_{selected_finetuning}_{augmentation}_seed{seed}"
            plans.append({"run_id": run_id, "config": config})
show_table([{"run_id": p["run_id"], "augmentation": p["config"]["augmentation"],
             "model": p["config"]["model"]["name"], "finetuning": p["config"]["model"]["finetuning"],
             "seed": p["config"]["seed"], "lr": p["config"]["training"]["lr"]}
            for p in plans])

In [ ]:
if not plans:
    print("尚未完成前置模型与策略选择，暂不训练。")
elif not RUN_TRAINING:
    print("当前只展示计划。确认后将 RUN_TRAINING 设为 True。")
elif not LABELS.get("classes"):
    print("请先完成共同类别及固定划分。")
else:
    from src.train import run_experiment
    for plan in plans:
        run_dir = run_experiment(PROJECT_ROOT, plan["config"], plan["run_id"])
        print("已保存：", run_dir)

## 2. 可选：小规模超参数实验

只有在验证结果支持继续研究时才增加 E4。先根据 E3 验证集选定增强，再填写 `E4_AUGMENTATION` 和候选学习率。E4 只改变学习率，其他设置固定，并使用单独的实验编号。不要查看 PlantDoc 后再确定候选范围。

In [ ]:
RUN_E4_TRAINING = False
E4_AUGMENTATION = None  # 依据 E3 验证结果填写：none / basic / combined
E4_LEARNING_RATES = []  # 例如填写两个有依据的候选学习率；空列表表示不做 E4

e4_plans = []
if selected_backbone and selected_finetuning and E4_AUGMENTATION and E4_LEARNING_RATES:
    for learning_rate in E4_LEARNING_RATES:
        for seed in EXPERIMENTS.get("seeds", [BASE.get("seed", 42)]):
            config = deepcopy(BASE)
            config["model"]["name"] = selected_backbone
            config["model"]["finetuning"] = selected_finetuning
            config["augmentation"] = E4_AUGMENTATION
            config["training"]["lr"] = float(learning_rate)
            config["seed"] = int(seed)
            lr_tag = f"{float(learning_rate):g}".replace(".", "p")
            run_id = f"e4_{selected_backbone}_{E4_AUGMENTATION}_lr{lr_tag}_seed{seed}"
            e4_plans.append({"run_id": run_id, "config": config})
show_table([{"run_id": p["run_id"], "lr": p["config"]["training"]["lr"],
             "augmentation": p["config"]["augmentation"]} for p in e4_plans])
if RUN_E4_TRAINING and e4_plans and LABELS.get("classes"):
    from src.train import run_experiment
    for plan in e4_plans:
        print("已保存：", run_experiment(PROJECT_ROOT, plan["config"], plan["run_id"]))
else:
    print("E4 未启动；不需要额外参数研究时可保持当前设置。")

In [ ]:
show_table(summarize_validation(plans + e4_plans))
print("仅依据验证集比较策略；最终外部测试统一在 Notebook 05 进行。")


## 结论记录

- 各增强策略的验证指标、曲线与波动：待填写。
- 是否出现过强增强或过拟合：待填写。
- 选定增强及依据：待填写。
- E4 的必要性、搜索范围与结果（如开展）：待填写。
- 准备提交 Notebook 05 的完整候选实验列表：待填写，并在外测前锁定。